This notebook contains cells that download statewide monthly NOAA Climate at a Glance data.
This notebook was written by Becky Bolinger, 2026, with Gemini Code Assist.

Data can be found at the base url: https://www.ncei.noaa.gov/access/monitoring/climate-at-a-glance/statewide/time-series/

When run, this will grab all states, all months from 1895 to the most recent month. Data is stored and archived in a netcdf file. The most current data is also contained in four csv files, one for each variable - average temperature, maximum temperature, minimum temperature, and precipitation.

The following cell reads the most recently updated data available at www.ncei.noaa.gov. Four variables are saved in a dataset and output to a netcdf file and four csv files. Two separate python files are required to successfully run the cell:

states.py - contains the list of state names and their associated numbers for identification in the url.
export_flat_ds.py - this code takes the dataset and modifies it to be exported into the four csv files.

In [3]:
import requests
import pandas as pd
import io
import xarray as xr
import numpy as np
from datetime import datetime
from states import state_dict

# Get the current year for the url to the data
yr = datetime.now().strftime('%Y')
# yr = '2026'

# Define the base URL
base_url = "https://www.ncei.noaa.gov/access/monitoring/climate-at-a-glance/statewide/time-series/{state_num}/{variable}/1/0/1895-{yr}/data.csv"

# Define the variables to download
variables = ["tavg", "tmax", "tmin", "pcp"]

# Define the state numbers (1-51, excluding 49)
state_numbers = list(range(1, 52))
state_numbers.remove(49)

# Create a dictionary to store the data for each state and variable
data = {}

# Loop through the state numbers and variables
for state_num in state_numbers:
    data[state_num] = {}
    for variable in variables:
        # Construct the URL
        url = base_url.format(state_num=state_num, variable=variable, yr=yr)

        # Download the CSV file
        response = requests.get(url)
        response.raise_for_status()  # Raise an exception for bad status codes

        csv_data = response.content.decode('utf-8')

        state_name = state_dict[state_num]

        # Read the CSV file into a pandas DataFrame
        df = pd.read_csv(io.StringIO(csv_data), header=None, skiprows=3)
        df.columns = ['Date', 'Value']

        # Convert the 'Date' column to datetime objects
        df['Date'] = pd.to_datetime(df['Date'], format='%Y%m')

        # Set the 'Date' column as the index
        df.set_index('Date', inplace=True)

        # Store the data in the dictionary
        data[state_num][variable] = df['Value']
        data[state_num][variable].name = state_name  # Store state name

# Create an xarray Dataset
ds = xr.Dataset()

# Add the data to the Dataset
for state_num in state_numbers:
    for variable in variables:
        # Convert pandas Series to xarray DataArray
        data[state_num][variable].index.name = 'time'  # Rename index to 'time'
        ds_state_name = data[state_num][variable].name
        da = xr.DataArray(
            data[state_num][variable],
            dims=['time'],
            coords={'time': data[state_num][variable].index},
            attrs={'state': ds_state_name, 'variable': variable, 'state_number': state_num}
        )
        ds[f'{ds_state_name}_{variable}'] = da


# Export to CSV files for each variable
from export_flat_ds import export_csvs_from_ds
import os

local_data_dir = os.path.join(os.getcwd(), 'data')
os.makedirs(local_data_dir, exist_ok=True)
export_csvs_from_ds(ds, output_dir=local_data_dir)

# Save the Dataset to a netCDF file
output_dir = os.path.join(local_data_dir, 'statewide_climate_data.nc')
ds.to_netcdf(output_dir)



Exporting tavg data...
  Saved to /Users/bbolinger/Desktop/DROUGHT/programming/git_projects/cag_data_pipeline/data/cag_tavg_data.csv
Exporting tmin data...
  Saved to /Users/bbolinger/Desktop/DROUGHT/programming/git_projects/cag_data_pipeline/data/cag_tmin_data.csv
Exporting tmax data...
  Saved to /Users/bbolinger/Desktop/DROUGHT/programming/git_projects/cag_data_pipeline/data/cag_tmax_data.csv
Exporting pcp data...
  Saved to /Users/bbolinger/Desktop/DROUGHT/programming/git_projects/cag_data_pipeline/data/cag_pcp_data.csv


The following cell archives the most recently generated netcdf file of the data into an archive folder. The most recent file should always be used to ensure the highest quality data. Archives are kept in case of corrupt or altered data being written to the current file.

In [4]:
import shutil
import os
from datetime import datetime

# Configuration
input_path = output_dir  # Path to the netcdf file we want to archive, defined in previous cell
archive_dir = os.path.join(os.getcwd(), 'archive')

def main():
    # Check if input file exists
    if not os.path.exists(input_path):
        print(f"Error: Input file not found at {input_path}")
        return

    # Create archive directory if it doesn't exist
    if not os.path.exists(archive_dir):
        try:
            os.makedirs(archive_dir)
            print(f"Created archive directory: {archive_dir}")
        except OSError as e:
            print(f"Error creating archive directory: {e}")
            return

    # Generate timestamp YYYYMMDD
    timestamp = datetime.now().strftime('%Y%m%d')

    # Construct destination filename (e.g., statewide_climate_data_20231027.nc)
    filename = os.path.basename(input_path)
    name, ext = os.path.splitext(filename)
    new_filename = f"{name}_{timestamp}{ext}"
    destination = os.path.join(archive_dir, new_filename)

    # Copy file
    try:
        shutil.copy2(input_path, destination)
        print(f"Successfully archived file to: {destination}")
    except Exception as e:
        print(f"Error copying file: {e}")

if __name__ == '__main__':
    main()



Successfully archived file to: /Users/bbolinger/Desktop/DROUGHT/programming/git_projects/cag_data_pipeline/archive/statewide_climate_data_20260326.nc


The following code reads the most recent netcdf file and gives a summary of the information contained within.

In [5]:
import netCDF4 as nc

file_path = output_dir # path to netcdf file we're reading, defined in the first cell

def print_nc_structure(path):
    try:
        with nc.Dataset(path, 'r') as ds:
            print(f"File Format: {ds.file_format}")
            
            print("\nDimensions:")
            for name, dim in ds.dimensions.items():
                print(f"  {name}: {dim.size}")
            
            print("\nVariables:")
            for name, var in ds.variables.items():
                print(f"  {name} {var.shape} ({var.dtype})")
                for attr in var.ncattrs():
                    print(f"    {attr}: {getattr(var, attr)}")
            
            print("\nGlobal Attributes:")
            for attr in ds.ncattrs():
                print(f"  {attr}: {getattr(ds, attr)}")
    except Exception as e:
        print(f"Error reading file: {e}")

if __name__ == "__main__":
    print_nc_structure(file_path)


File Format: NETCDF4

Dimensions:
  time: 1574

Variables:
  time (1574,) (int64)
    units: days since 1895-01-01 00:00:00
    calendar: proleptic_gregorian
  Alabama_tavg (1574,) (float64)
    _FillValue: nan
    state: Alabama
    variable: tavg
    state_number: 1
  Alabama_tmax (1574,) (float64)
    _FillValue: nan
    state: Alabama
    variable: tmax
    state_number: 1
  Alabama_tmin (1574,) (float64)
    _FillValue: nan
    state: Alabama
    variable: tmin
    state_number: 1
  Alabama_pcp (1574,) (float64)
    _FillValue: nan
    state: Alabama
    variable: pcp
    state_number: 1
  Arizona_tavg (1574,) (float64)
    _FillValue: nan
    state: Arizona
    variable: tavg
    state_number: 2
  Arizona_tmax (1574,) (float64)
    _FillValue: nan
    state: Arizona
    variable: tmax
    state_number: 2
  Arizona_tmin (1574,) (float64)
    _FillValue: nan
    state: Arizona
    variable: tmin
    state_number: 2
  Arizona_pcp (1574,) (float64)
    _FillValue: nan
    state: Ariz

The following cell is a scratch pad for checking data and variables.

In [6]:
state_num = 5;
variable = "tavg";

data[state_num][variable].index.name = 'time'
da = xr.DataArray(
    data[state_num][variable],
    dims=['time'],
    coords={'time': data[state_num][variable].index},
    attrs={'state': data[state_num][variable].name, 'variable': variable}
)

#ds = xr.Dataset()
ds[state_num] = da

data[37][variable].name


'Rhode Island'